# Test accuracy on duplicated videos

| EXP NO    | Model    | Split            |
|-----------|----------|------------------|
| exp000    | MViTv2_S | asl2000_cutoff_9 |

WLASL-2000 contains identical videos filed under several glosses, some of them split across
train and test. This notebook asks how the model does on test videos that have an identical copy
elsewhere in the dataset, split by where the copy is and whether it carries the same gloss.

Duplicate groups come from
[`find_duplicates.py`](../dataset_analysis/find_duplicates.py) (exact duplicates only: every
decoded frame identical; its near-duplicate check gives false positives). Top-k predictions are
the stash from [all_misspredictions.ipynb](./all_misspredictions.ipynb). Dataset fields and
gotchas: [WLASL_info.md](../../info/WLASL_info.md).

In [1]:
from typing import Literal, TypeAlias

import pandas as pd
from IPython.display import display
from pydantic import TypeAdapter

from src.configs import get_class_list
from src.results import format_exp_label, get_out_stub, get_stash_path, load_json
from src.results.dataset_analysis.find_duplicates import DEFAULT_REPORT, Member, Report
from src.run_types import AVAIL_SETS, CUTOFF_SPLITS, InstanceTopK

## Setup

In [2]:
split_name: CUTOFF_SPLITS = 'asl2000_cutoff_9'
set_name: AVAIL_SETS = 'test' # the set the predictions were stashed for
model_name = 'MViTv2_S'
exp_no = '000'
checkpoint_num = None
top_n = 5 # the second accuracy reported, alongside top-1

stub = get_out_stub(split_name, model_name, format_exp_label(exp_no), checkpoint_num)
classes = get_class_list()
predictions = TypeAdapter(list[InstanceTopK]).validate_python(
    load_json(get_stash_path(stub, base_name='instance_topk'))
)
report = Report.model_validate_json(DEFAULT_REPORT.read_text())
print(f'{len(predictions)} {set_name} predictions; duplicate report for {report.num_instances} instances')

2879 test predictions; duplicate report for 21095 instances


## Where each test video's copies are

The report was made from the plain `asl2000` split, while the model was trained on
`asl2000_cutoff_9`. The cutoff removed three train/val videos (see WLASL_info), none of which is
in a duplicate group, so the sets match; the assert checks this.

In [3]:
CUTOFF_REMOVED = {'18223', '59958', '15144'}

copies_of: dict[str, list[Member]] = {}
for group in report.groups:
    if group.kind != 'exact':
        continue
    assert not CUTOFF_REMOVED & {m.video_id for m in group.members}
    for member in group.members:
        copies_of[member.video_id] = [m for m in group.members if m.video_id != member.video_id]

CopyKind: TypeAlias = Literal[
    'no copy',
    'train copy, other gloss',
    'train copy, same gloss',
    'train copy, same and other gloss',
    'val copy only',
    'test copy only',
]


def copy_kind(gloss: str, copies: list[Member]) -> CopyKind:
    """Classify a video by its copies: training copies take precedence over val, then test."""
    train_glosses = {m.gloss for m in copies if m.set == 'train'}
    if train_glosses:
        same, other = gloss in train_glosses, bool(train_glosses - {gloss})
        if same and other:
            return 'train copy, same and other gloss'
        return 'train copy, same gloss' if same else 'train copy, other gloss'
    if any(m.set == 'val' for m in copies):
        return 'val copy only'
    return 'test copy only' if copies else 'no copy'


rows: list[dict[str, object]] = []
for pred in predictions:
    gloss = classes[pred.target]
    copies = copies_of.get(pred.video_id, [])
    train_other = {m.gloss for m in copies if m.set == 'train'} - {gloss}
    top = [classes[i] for i in pred.topk_idxs]
    rows.append({
        'video_id': pred.video_id,
        'gloss': gloss,
        'kind': copy_kind(gloss, copies),
        'predicted': top[0],
        'confidence': pred.topk_probs[0],
        'top1': top[0] == gloss,
        f'top{top_n}': gloss in top[:top_n],
        'pred_is_train_copy_gloss': top[0] in train_other,
        f'train_copy_gloss_in_top{top_n}': bool(train_other & set(top[:top_n])),
        'copies': ', '.join(f'{m.video_id} {m.gloss}/{m.set}' for m in copies),
    })
df = pd.DataFrame(rows)
assert df['video_id'].is_unique

## Accuracy by kind of copy (%)

In [4]:
kind_order = list(CopyKind.__args__)  # pyright: ignore[reportAttributeAccessIssue]
summary = (
    df.groupby('kind')
    .agg(
        n=('video_id', 'size'),
        top1=('top1', 'mean'),
        **{f'top{top_n}': (f'top{top_n}', 'mean')},
        pred_is_train_copy_gloss=('pred_is_train_copy_gloss', 'mean'),
        **{f'train_copy_gloss_in_top{top_n}': (f'train_copy_gloss_in_top{top_n}', 'mean')},
    )
    .reindex([k for k in kind_order if k in set(df['kind'])])
)
pct_cols = [c for c in summary.columns if c != 'n']
summary[pct_cols] = (summary[pct_cols] * 100).round(2)
summary

,n,top1,top5,pred_is_train_copy_gloss,train_copy_gloss_in_top5
kind,,,,,
no copy,2689,40.87,71.07,0.0,0.0
"train copy, other gloss",147,0.00,79.59,100.0,100.0
"train copy, same gloss",8,100.00,100.00,0.0,0.0
"train copy, same and other gloss",1,100.00,100.00,0.0,100.0
val copy only,28,35.71,78.57,0.0,0.0
test copy only,6,16.67,50.00,0.0,0.0


## Overall top-1 with and without the duplicated test videos (%)

In [5]:
def top1_pct(mask: 'pd.Series[bool]') -> float:
    return round(float(df.loc[mask, 'top1'].mean()) * 100, 2)


every = pd.Series(True, index=df.index)
conflicting = df['kind'].isin(['train copy, other gloss'])
any_copy = df['kind'] != 'no copy'
overall = pd.DataFrame(
    {
        'n': [int(every.sum()), int((~conflicting).sum()), int((~any_copy).sum())],
        'top1': [top1_pct(every), top1_pct(~conflicting), top1_pct(~any_copy)],
    },
    index=[
        f'all {set_name} videos (as reported)',
        'without videos whose only training copies have another gloss',
        'without any duplicated video',
    ],
)
overall

,n,top1
all test videos (as reported),2879,38.87
without videos whose only training copies have another gloss,2732,40.96
without any duplicated video,2689,40.87


## Test videos whose training copy has another gloss

The model's prediction against the copy's gloss, for every such video.

In [6]:
conflicts = df.loc[conflicting, ['video_id', 'gloss', 'predicted', 'confidence', 'copies']]
print(f'{len(conflicts)} videos; prediction is a training copy\'s gloss for '
      f'{int(df.loc[conflicting, "pred_is_train_copy_gloss"].sum())}')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(conflicts.sort_values('gloss').reset_index(drop=True))

147 videos; prediction is a training copy's gloss for 147


,video_id,gloss,predicted,confidence,copies
0,02128,a lot,much,0.974842,37198 much/train
1,02048,allow,let,0.990067,32916 let/train
2,02049,allow,permit,0.987743,42142 permit/train
3,02544,angel,fairy,0.999388,20838 fairy/train
4,02545,angel,fairy,0.997564,20839 fairy/train
5,02701,another,other,0.573281,"18858 else/train, 40317 other/train"
6,02715,answer,reply,0.534065,"47345 reply/train, 47362 report/train"
7,03125,approve,take,0.965768,"00600 accept/val, 01159 adopt/val, 56700 take/train"
8,03278,argue,quarrel,0.939700,45656 quarrel/train
9,04429,awful,terrible,0.993457,57489 terrible/train


## Identical test videos with different glosses

Pairs like these can't both be right: the input is the same, the targets differ.

In [7]:
test_conflicts = df[
    df['copies'].str.contains('/test')
    & df.apply(
        lambda r: any(
            m.set == 'test' and m.gloss != r['gloss'] for m in copies_of.get(str(r['video_id']), [])
        ),
        axis=1,
    )
]
print(f'{len(test_conflicts)} test videos have an identical test video with another gloss')
test_conflicts[['video_id', 'gloss', 'predicted', 'top1', 'copies']].sort_values('video_id')

14 test videos have an identical test video with another gloss


,video_id,gloss,predicted,top1,copies
1351,03425,arrest,hit,False,09468 catch/test
16,05741,before,past,False,"01556 ago/train, 32253 last/test, 41452 past/t..."
598,09468,catch,hit,False,03425 arrest/test
2225,12217,complete,sue,False,19114 end/test
1090,14839,dead,death,False,"14955 death/train, 16164 die/test"
1968,16164,die,death,False,"14839 dead/test, 14955 death/train"
1116,19114,end,sue,False,12217 complete/test
178,32253,last,past,False,"01556 ago/train, 05741 before/test, 41452 past..."
2076,46333,realize,realize,True,46433 reason/test
1231,46374,really,true,False,"46317 real/train, 56035 sure/test, 59795 true/..."
